# Train the ingredient model on Colab
Before running: upload `merged-v3.zip` to the top of your Google Drive (on your Mac: `cd VisionModel/datasets && zip -r -0 merged-v3.zip merged-v3`).

Runtime → Change runtime type → **GPU**. Then run the cells top to bottom.

In [ ]:
# Settings: edit these
DATA_ZIP = "/content/drive/MyDrive/merged-v3.zip"
RUN_DIR  = "/content/drive/MyDrive/runs/v3-colab"   # on Drive so it survives a disconnect
EPOCHS   = 50   # ~7 h on a 3060; a free T4 is slower and sessions cap at ~12 h, so consider 40

## 1. GPU + Drive

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

## 2. Code and dependencies

In [ ]:
!git clone --depth 1 -b VisionModel https://github.com/ArunSisarran/Capstone-Project.git /content/Capstone-Project
%cd /content/Capstone-Project/VisionModel
!pip install -q ultralytics

## 3. Dataset
Copied to local disk: reading 56k small files straight from Drive is very slow. `data.yaml` holds the Linux machine's absolute path, so it's rewritten here.

In [ ]:
!cp "$DATA_ZIP" /content/data.zip
!unzip -q -o /content/data.zip -d datasets/
!sed -i 's#^path: .*#path: /content/Capstone-Project/VisionModel/datasets/merged-v3#' datasets/merged-v3/data.yaml
!head -3 datasets/merged-v3/data.yaml

## 4. Smoke test (~1 min)

In [ ]:
!python model.py train --data datasets/merged-v3/data.yaml --out runs/smoke --epochs 1 --max-steps 20 --workers 2

## 5. Full training
`model.py` can't resume: if Colab disconnects, `last.pt` in `RUN_DIR` holds progress so far, but a rerun starts from epoch 1.

In [ ]:
!python model.py train --data datasets/merged-v3/data.yaml --out "$RUN_DIR" --epochs $EPOCHS --workers 2

## 6. Score on the test split

In [ ]:
!yolo val model="$RUN_DIR/best.pt" data=datasets/merged-v3/data.yaml split=test project=/content/scratch